<a href="https://colab.research.google.com/github/SkyWarLimit/PemMes_17/blob/main/JS06/JS06_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
import glob
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
# Load dan Preprocessing Data Voice
# Memuat dataset
df_voice = pd.read_csv('voice (1).csv')

# Mengasumsikan kolom target/label bernama 'label'
X_voice = df_voice.drop('label', axis=1)
y_voice = df_voice['label']

# Melakukan encoding pada label jika berupa string ('male' / 'female')
le = LabelEncoder()
y_voice = le.fit_transform(y_voice)

# Melakukan standarisasi (Scaling) fitur karena SVM sangat sensitif terhadap skala data
scaler = StandardScaler()
X_voice_scaled = scaler.fit_transform(X_voice)

print("Dimensi fitur data voice:", X_voice_scaled.shape)

Dimensi fitur data voice: (3168, 20)


In [ ]:
# Training, Evaluasi, dan Tabulasi SVM
splits = {'70:30': 0.3, '80:20': 0.2}
kernels = ['linear', 'poly', 'rbf']
hasil_evaluasi = []

# Iterasi untuk setiap rasio split
for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X_voice_scaled, y_voice, test_size=test_size, random_state=42
    )

    # Iterasi untuk setiap kernel
    for kernel in kernels:
        model = SVC(kernel=kernel)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        akurasi = accuracy_score(y_test, y_pred)

        # Menyimpan hasil ke dalam list
        hasil_evaluasi.append({
            'Rasio Split': split_name,
            'Kernel': kernel,
            'Akurasi': akurasi
        })

# Membuat DataFrame untuk menabulasikan hasil
df_hasil = pd.DataFrame(hasil_evaluasi)
print("--- Tabulasi Performansi Akurasi SVM (Data Voice) ---")
display(df_hasil)

--- Tabulasi Performansi Akurasi SVM (Data Voice) ---


,Rasio Split,Kernel,Akurasi
0,70:30,linear,0.970557
1,70:30,poly,0.956887
2,70:30,rbf,0.981073
3,80:20,linear,0.976341
4,80:20,poly,0.968454
5,80:20,rbf,0.982650


In [ ]:
# Ekstraksi File ZIP dan Fitur Histogram Citra Siang & Malam
import cv2
import os
import glob
import numpy as np
import zipfile

# 1. Tentukan nama file ZIP Anda
zip_path = 'images.zip'
extract_path = './'

if os.path.exists(zip_path):
    print(f"Mengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Ekstraksi berhasil!")
else:
    print(f"Peringatan: File {zip_path} tidak ditemukan. Jika folder 'images' sudah ada, proses akan dilanjutkan.")

# 2. Fungsi Ekstraksi Fitur Histogram
def ekstrak_histogram(image_path):
    # Membaca citra dan konversi BGR ke HSV
    image = cv2.imread(image_path)
    if image is None:
        return None
    image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Menghitung histogram (menggunakan channel V/Value untuk pencahayaan siang/malam)
    hist = cv2.calcHist([image_hsv], [2], None, [256], [0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

data_citra = []
label_citra = []

# Base folder sesuai hasil ekstraksi
base_dir = 'images'
kategori = ['day', 'night']

# 3. Loop untuk membaca folder test dan training
for sub_dir in ['training', 'test']:
    for label in kategori:
        # Menyatukan path: misal images/training/day
        folder_path = os.path.join(base_dir, sub_dir, label)

        if os.path.exists(folder_path):
            # Mengambil semua file berformat .jpg
            for img_file in glob.glob(os.path.join(folder_path, '*.jpg')):
                hist_fitur = ekstrak_histogram(img_file)
                if hist_fitur is not None:
                    data_citra.append(hist_fitur)
                    # Label: 1 untuk 'day', 0 untuk 'night'
                    label_citra.append(1 if label == 'day' else 0)
        else:
            print(f"Folder tidak ditemukan: {folder_path}")

X_img = np.array(data_citra)
y_img = np.array(label_citra)

print("-" * 30)
print(f"Total citra berhasil diproses: {len(X_img)}")

Mengekstrak images.zip...
Ekstraksi berhasil!
------------------------------
Total citra berhasil diproses: 400


In [ ]:
# SVM Kernel RBF dengan Hyperparameter Tuning
if len(X_img) > 0:
    # Split data rasio 80:20
    X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
        X_img, y_img, test_size=0.2, random_state=42
    )

    # Menentukan grid hyperparameter untuk dicari kombinasinya
    param_grid = {
        'C': [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.1, 0.01],
        'kernel': ['rbf']
    }

    print("Memulai pencarian hyperparameter terbaik untuk kernel RBF...")
    # cv=5 berarti menggunakan 5-fold cross validation
    grid_search = GridSearchCV(SVC(), param_grid, refit=True, verbose=1, cv=5)
    grid_search.fit(X_train_img, y_train_img)

    print("\nParameter RBF terbaik ditemukan:", grid_search.best_params_)

    # Memprediksi data uji dengan model terbaik
    y_pred_img = grid_search.predict(X_test_img)
    akurasi_img = accuracy_score(y_test_img, y_pred_img)

    print("-" * 50)
    print(f"Akurasi Model Klasifikasi Siang/Malam (Split 80:20): {akurasi_img * 100:.2f}%")
else:
    print("Dataset citra kosong. Silakan periksa path atau folder ekstraksi 'images' Anda.")

Memulai pencarian hyperparameter terbaik untuk kernel RBF...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Parameter RBF terbaik ditemukan: {'C': 100, 'gamma': 0.1, 'kernel': 'rbf'}
--------------------------------------------------
Akurasi Model Klasifikasi Siang/Malam (Split 80:20): 100.00%
